<a href="https://colab.research.google.com/github/Savithmee/Deep-Learning-CW-/blob/Sahanmee/EntailmentPreprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install spacy nltk psutil
import nltk
nltk.download('punkt_tab', quiet=True); nltk.download('punkt', quiet=True)

True

In [2]:
import os, json
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/Deep Learning Coursework'
PASSAGES = f'{DATA_DIR}/passages_bm25_ChatGPT_humfacts.jsonl'
LABELS   = f'{DATA_DIR}/dev_labeled_ChatGPT.jsonl'

# quick check that both files are found
print(os.listdir(DATA_DIR))
assert os.path.exists(PASSAGES), 'passages file not found, check the name in the list above'
assert os.path.exists(LABELS),   'labels file not found, check the name in the list above'

Mounted at /content/drive
['dev_labeled_ChatGPT.jsonl', 'passages_bm25_ChatGPT_humfacts.jsonl']


In [3]:
%%writefile preprocessing.py
import re, html, unicodedata
from typing import List, Dict, Optional

STOPWORDS = set("""a about above after again against all am an and any are as at be because been before being
below between both but by can did do does doing down during each few for from further had has have having he her
here hers herself him himself his how i if in into is it its itself just me more most my myself no nor not now of
off on once only or other our ours ourselves out over own same she should so some such than that the their theirs
them themselves then there these they this those through to too under until up very was we were what when where
which while who whom why will with you your yours yourself yourselves also""".split())

DEFAULT_CFG = dict(
    splitter="regex",        # tags | naive | regex | nltk | spacy
    min_words=3,
    max_words=100,
    window=1,                # 1 = single sentences, 2/3 = windows, 10**6 = whole passage
    stride=1,
    title="none",            # none | all | pronoun
    prune_thr=None,          # None = no pruning
    min_keep=1,
    max_keep=None,
    order="window_then_prune",   # or prune_then_window
)

# cleaning
_TAG = re.compile(r"<[^>]+>")
_CITE = re.compile(r"\[\d+\]|\[citation needed\]", re.I)

def clean_text(text: str, title: Optional[str] = None) -> str:
    t = html.unescape(text)
    t = unicodedata.normalize("NFKC", t)
    t = _CITE.sub(" ", t)
    t = _TAG.sub(" ", t)
    t = re.sub(r"\s+", " ", t).strip()
    if title and t.startswith(title + " " + title):
        t = t[len(title) + 1:]
    return t

# sentence splitting
_REGEX_SPLIT = re.compile(r"(?<=[.!?])\s+(?=[\"'(A-Z0-9])")
_SPACY = None
_NLTK_OK = None

def _split_text(segments: List[str], splitter: str) -> List[str]:
    global _SPACY, _NLTK_OK
    if splitter == "tags":
        return segments
    out = []
    if splitter == "naive":
        for s in segments:
            parts = s.split(". ")
            out += [p if p.endswith(".") or i == len(parts) - 1 else p + "." for i, p in enumerate(parts)]
    elif splitter == "regex":
        for s in segments:
            out += _REGEX_SPLIT.split(s)
    elif splitter == "nltk":
        import nltk
        from nltk.tokenize import sent_tokenize
        if _NLTK_OK is None:
            nltk.download("punkt_tab", quiet=True)
            nltk.download("punkt", quiet=True)
            _NLTK_OK = True
        for s in segments:
            out += sent_tokenize(s)
    elif splitter == "spacy":
        if _SPACY is None:
            import spacy
            _SPACY = spacy.blank("en")
            _SPACY.add_pipe("sentencizer")
        for doc in _SPACY.pipe(segments):
            out += [x.text for x in doc.sents]
    else:
        raise ValueError("unknown splitter " + splitter)
    return out

def _chunk(text: str, max_words: Optional[int]) -> List[str]:
    words = text.split()
    if not max_words or len(words) <= max_words:
        return [text]
    pieces, cur = [], []
    for part in re.split(r"(?<=;)\s+", text):
        pw = part.split()
        if cur and len(cur) + len(pw) > max_words:
            pieces.append(" ".join(cur)); cur = []
        cur += pw
    if cur:
        pieces.append(" ".join(cur))
    out = []
    for p in pieces:
        w = p.split()
        out += [" ".join(w[i:i + max_words]) for i in range(0, len(w), max_words)]
    return out

def sentence_units(passages: List[dict], splitter="regex", min_words=3, max_words=100) -> List[Dict]:
    units, seen = [], set()
    for pid, p in enumerate(passages):
        title = p.get("title", "")
        segs = [clean_text(s, title) for s in re.split(r"</?s>", p["text"])]
        segs = [s for s in segs if s]
        sid = 0
        for sent in _split_text(segs, splitter):
            for piece in _chunk(sent.strip(), max_words):
                if len(piece.split()) < min_words:
                    continue
                key = piece.lower()
                if key in seen:
                    continue
                seen.add(key)
                units.append(dict(title=title, pid=pid, sid=sid, text=piece))
                sid += 1
    return units

def _windows(units: List[Dict], window: int, stride: int) -> List[Dict]:
    if window <= 1:
        return [dict(title=u["title"], text=u["text"]) for u in units]
    out, seen = [], set()
    pids = sorted({u["pid"] for u in units})
    for pid in pids:
        g = [u for u in units if u["pid"] == pid]
        n = len(g)
        starts = [0] if window >= n else list(range(0, n - window + 1, max(1, stride)))
        if window < n and starts[-1] != n - window:
            starts.append(n - window)
        for s in starts:
            text = " ".join(u["text"] for u in g[s:s + window])
            if text.lower() not in seen:
                seen.add(text.lower())
                out.append(dict(title=g[0]["title"], text=text))
    return out

def candidates(passages: List[dict], cfg: Optional[dict] = None) -> List[Dict]:
    c = {**DEFAULT_CFG, **(cfg or {})}
    units = sentence_units(passages, c["splitter"], c["min_words"], c["max_words"])
    return _windows(units, c["window"], c["stride"])

# overlap + pruning
def content_tokens(text: str) -> set:
    return {w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOPWORDS}

def overlap_score(fact_tokens: set, text: str) -> float:
    if not fact_tokens:
        return 1.0
    return len(fact_tokens & content_tokens(text)) / len(fact_tokens)

def select(scores: List[float], min_overlap: float, min_keep: int = 1, max_keep: Optional[int] = None) -> List[int]:
    if not scores:
        return []
    keep = [i for i, s in enumerate(scores) if s >= min_overlap]
    if max_keep is not None and len(keep) > max_keep:
        keep = sorted(sorted(keep, key=lambda i: -scores[i])[:max_keep])
    if not keep:
        keep = sorted(sorted(range(len(scores)), key=lambda i: -scores[i])[:max(1, min_keep)])
    return keep

def prune(fact: str, sentences: List[str], min_overlap: float = 0.1,
          min_keep: int = 1, max_keep: Optional[int] = None) -> List[str]:
    ft = content_tokens(fact)
    scores = [overlap_score(ft, s) for s in sentences]
    return [sentences[i] for i in select(scores, min_overlap, min_keep, max_keep)]

# titles + full pipeline
_PRON = re.compile(r"^(He|She|They|His|Her|Their|It|Its)\b")

def _with_title(title: str, text: str, mode: str) -> str:
    if mode == "all" or (mode == "pronoun" and _PRON.match(text)):
        return f"{title}: {text}"
    return text

def make_premises(fact: str, passages: List[dict], cfg: Optional[dict] = None) -> List[Dict]:
    c = {**DEFAULT_CFG, **(cfg or {})}
    ft = content_tokens(fact)
    thr = c["prune_thr"]
    if c["order"] == "prune_then_window" and c["window"] > 1 and thr is not None:
        units = sentence_units(passages, c["splitter"], c["min_words"], c["max_words"])
        sc = [overlap_score(ft, u["text"]) for u in units]
        items, seen = [], set()
        for i in select(sc, thr, c["min_keep"], c["max_keep"]):
            u = units[i]
            grp = [v for v in units if v["pid"] == u["pid"] and u["sid"] <= v["sid"] < u["sid"] + c["window"]]
            plain = " ".join(v["text"] for v in grp)
            if plain.lower() not in seen:
                seen.add(plain.lower())
                items.append(dict(title=u["title"], text=plain))
        cands = items
        scores = [overlap_score(ft, x["text"]) for x in cands]
        keep = list(range(len(cands)))
    else:
        cands = candidates(passages, c)
        scores = [overlap_score(ft, x["text"]) for x in cands]
        keep = list(range(len(cands))) if thr is None else select(scores, thr, c["min_keep"], c["max_keep"])
    return [dict(text=_with_title(cands[i]["title"], cands[i]["text"], c["title"]),
                 plain=cands[i]["text"], overlap=scores[i]) for i in keep]

def split_passages(passages: List[dict], **cfg) -> List[str]:
    c = {**DEFAULT_CFG, **cfg}
    return [_with_title(x["title"], x["text"], c["title"]) for x in candidates(passages, c)]

Writing preprocessing.py


In [4]:
import sys, time, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, '.')
import importlib, preprocessing
importlib.reload(preprocessing)
from preprocessing import *

class FactExample:
    def __init__(self, fact, passages, label):
        self.fact, self.passages, self.label = fact, passages, label

def read_passages(path):
    d = {}
    for line in open(path):
        r = json.loads(line)
        d[r['sent']] = r['passages']
    return d

def read_fact_examples(path, f2p):
    ex = []
    for line in open(path):
        r = json.loads(line)
        for sent in (r['annotations'] or []):
            for fact in (sent['human-atomic-facts'] or []):
                if fact['text'] in f2p:
                    ex.append(FactExample(fact['text'], f2p[fact['text']], fact['label']))
    return ex

f2p = read_passages(PASSAGES)
EXAMPLES = read_fact_examples(LABELS, f2p)
print(len(EXAMPLES), 'examples;', pd.Series([e.label for e in EXAMPLES]).value_counts().to_dict())
S_EXAMPLES = [e for e in EXAMPLES if e.label == 'S']

def gold(e): return 1 if e.label == 'S' else 0      # IR counts as NS

def cov_stats(exs, cfg):
    n, lens, cov, t0 = [], [], [], time.time()
    for e in exs:
        c = candidates(e.passages, cfg)
        n.append(len(c)); lens += [len(x['text'].split()) for x in c]
        if e.label == 'S':
            ft = content_tokens(e.fact)
            cov.append(max([overlap_score(ft, x['text']) for x in c] or [0.0]))
    return dict(premises_per_fact=np.mean(n), words_per_premise=np.mean(lens),
                pct_over_128_words=100 * np.mean(np.array(lens) > 128),
                coverage=np.mean(cov), seconds=time.time() - t0)

221 examples; {'S': 112, 'NS': 93, 'IR': 16}
